# 01. First look at the lap data
Run `scripts/fetch_laps.py` first so that `data/processed/laps/` has parquet files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from vista.models.tyre_baseline import add_features, clean_laps

files = sorted(Path("data/processed/laps").glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(len(files), "races,", len(raw), "laps")
raw.head()

## How many laps survive cleaning?
Laps are dropped for pit in/out, safety cars, inaccurate timing and deleted laps.

In [ ]:
clean = clean_laps(raw)
summary = pd.DataFrame({
    "all_laps": raw.groupby("RaceId").size(),
    "clean_laps": clean.groupby("RaceId").size(),
})
summary["kept_pct"] = (100 * summary["clean_laps"] / summary["all_laps"]).round(1)
summary

## Lap time delta against tyre age, per compound

In [ ]:
df = add_features(clean)
races = sorted(df["RaceId"].unique())
fig, axes = plt.subplots(1, len(races), figsize=(5 * len(races), 4), sharey=True, squeeze=False)
for ax, race in zip(axes[0], races):
    for comp, g in df[df["RaceId"] == race].groupby("Compound"):
        ax.scatter(g["TyreLife"], g["DeltaS"], s=6, alpha=0.4, label=comp)
    ax.set_title(race)
    ax.set_xlabel("Tyre life (laps)")
axes[0][0].set_ylabel("Lap time delta (s)")
axes[0][0].legend()
plt.tight_layout()

## Stint lengths by compound

In [ ]:
stints = (
    raw.groupby(["RaceId", "Driver", "Stint"])
    .agg(laps=("LapNumber", "count"), compound=("Compound", "first"))
    .reset_index()
)
stints.groupby(["RaceId", "compound"])["laps"].describe().round(1)